# 무역 법리 LLM 파인튜닝 (16k 토큰 · GGUF 4bit 저장)
### Gemma-4-E2B-it · QLoRA · Unsloth · Google Colab A100+

| 섹션 | 내용 |
|---|---|
| 0 | 환경 설정 및 드라이브 마운트 |
| 1 | 경로·학습 설정 |
| 2 | 데이터 로드 · Train/Val 분할 |
| 3 | 모델 로드 및 LoRA 설정 |
| 4 | 학습 실행 |
| 5 | 저장 — safetensor(어댑터) → 병합 16bit → 4bit GGUF |
| 6 | 샘플 추론 검증 |

## 저장 전략 (드라이브 용량 최적화)
```
/content/              ← Colab 로컬 (런타임 종료 시 자동 삭제)
  work/
    checkpoints/       ← 학습 중 임시 체크포인트 (드라이브 미저장)
    merged_16bit/      ← GGUF 변환용 임시 (드라이브 미저장)

/content/drive/MyDrive/finetune/output/
  adapter_{RUN_ID}/   ← ✅ 드라이브 저장: LoRA 어댑터 (safetensor, ~100MB)
  gguf_{RUN_ID}/      ← ✅ 드라이브 저장: q4_k_m GGUF 최종 파일
```

> **드라이브에 남기는 것**: 어댑터(safetensor) + GGUF 4bit 최종 파일  
> **드라이브에 저장 안 하는 것**: 체크포인트, 병합 16bit 풀웨이트 (용량 절약)


## 섹션 0. 환경 설정

In [1]:
# [1] 기존 충돌 패키지 제거
!pip uninstall -y torchao unsloth unsloth_zoo peft transformers

# [2] 최신 규격에 맞는 종속성 설치
# PEFT 0.16.0 이상을 만족하기 위해 torchao를 최신 버전으로 설치합니다.
!pip install --no-cache-dir torchao>=0.16.0
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir --upgrade --force-reinstall --no-deps unsloth unsloth_zoo

# [3] 기타 필수 라이브러리 (버전 고정 해제하여 최신 호환성 유지)
!pip install --no-cache-dir "transformers>=4.46.0" "datasets>=3.4.1" "trl>=0.18.2"
!pip install --no-cache-dir xformers accelerate bitsandbytes gguf protobuf

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-r8kl8iwt/unsloth_c4aaa1aa276a41229dafc1454e83362f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-r8kl8iwt/unsloth_c4aaa1aa276a41229dafc1454e83362f
  Resolved https://github.com/unslothai/unsloth.git to commit 739ebeea8241d125c1bb8fa401207e62dcacb326
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 213.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/5

In [1]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount("/content/drive")
print("✅ 드라이브 마운트 완료")


Mounted at /content/drive
✅ 드라이브 마운트 완료


## 섹션 1. 경로 · 학습 설정
> **여기만 수정하면 됩니다.**


In [18]:
import os
from pathlib import Path
from datetime import datetime

# ── [수정 필요] 경로 설정 ──────────────────────────────────────────
# 학습 데이터 JSONL 파일 경로 (드라이브 내)
JSONL_PATH = "/content/drive/MyDrive/finetune/Gemma_Finetune_gemma4_e2b_ultimate_final.jsonl"

# 드라이브 최종 결과물 저장 루트
DRIVE_OUTPUT = Path("/content/drive/MyDrive/finetune/output")

# ── [수정 가능] 모델 및 학습 설정 ─────────────────────────────────
MODEL_ID       = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 16384   # 16k 토큰 (A100 40GB 이상 필요)
                          # OOM 시: 8192 → 4096 순으로 낮추기
LOAD_IN_4BIT   = True

# LoRA 설정
LORA_RANK  = 32   # 법률 도메인 특화 — r=16보다 높은 표현력
LORA_ALPHA = 32   # 권장: alpha == r

# 학습/검증 분할
EVAL_RATIO = 0.05   # 전체의 5% 검증
EVAL_STEPS = 30     # 몇 스텝마다 검증

# 학습 하이퍼파라미터
BATCH_SIZE    = 1
GRAD_ACCUM    = 4      # effective batch = 4
LEARNING_RATE = 1e-4
MAX_STEPS     = None   # None이면 NUM_EPOCHS 사용
NUM_EPOCHS    = 5
WARMUP_STEPS  = 10
SEED          = 3407

# ── 경로 자동 생성 ────────────────────────────────────────────────
RUN_ID = datetime.now().strftime("%m%d_%H%M")

# ✅ 드라이브 저장 경로 (영구 보존)
ADAPTER_DIR = DRIVE_OUTPUT / f"adapter_{RUN_ID}"
GGUF_DIR    = DRIVE_OUTPUT / f"gguf_{RUN_ID}"

# 로컬 임시 경로 (드라이브 미저장 — 런타임 종료 시 자동 삭제)
LOCAL_WORK       = Path("/content/work")
CHECKPOINT_DIR   = LOCAL_WORK / f"checkpoints_{RUN_ID}"
MERGED_16BIT_DIR = LOCAL_WORK / "merged_16bit"
LOCAL_GGUF_DIR   = LOCAL_WORK / "gguf_tmp" # New local temp for GGUF

for d in [ADAPTER_DIR, GGUF_DIR, CHECKPOINT_DIR, MERGED_16BIT_DIR, LOCAL_GGUF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Run ID            : {RUN_ID}")
print(f"학습 데이터       : {JSONL_PATH}")
print()
print("=== 드라이브 저장 (영구 보존) ===")
print(f"  어댑터(safetensor): {ADAPTER_DIR}")
print(f"  GGUF 4bit         : {GGUF_DIR}")
print()
print("=== 로컬 임시 (런타임 종료 시 삭제) ===")
print(f"  체크포인트        : {CHECKPOINT_DIR}")
print(f"  병합 16bit        : {MERGED_16BIT_DIR}")
print(f"  GGUF 임시         : {LOCAL_GGUF_DIR}") # Print new local temp GGUF dir
print()
print(f"MAX_SEQ_LENGTH    : {MAX_SEQ_LENGTH:,} 토큰")
print(f"LORA_RANK / ALPHA : {LORA_RANK} / {LORA_ALPHA}")

Run ID            : 0515_0151
학습 데이터       : /content/drive/MyDrive/finetune/Gemma_Finetune_gemma4_e2b_ultimate_final.jsonl

=== 드라이브 저장 (영구 보존) ===
  어댑터(safetensor): /content/drive/MyDrive/finetune/output/adapter_0515_0151
  GGUF 4bit         : /content/drive/MyDrive/finetune/output/gguf_0515_0151

=== 로컬 임시 (런타임 종료 시 삭제) ===
  체크포인트        : /content/work/checkpoints_0515_0151
  병합 16bit        : /content/work/merged_16bit
  GGUF 임시         : /content/work/gguf_tmp

MAX_SEQ_LENGTH    : 16,384 토큰
LORA_RANK / ALPHA : 32 / 32


## 섹션 2. 데이터 로드 · 포맷 변환 · Train/Val 분할

**순서**: JSONL 로드 → `<|im_start|>` 파싱 → `conversations` 변환 → `apply_chat_template` → 분할

Gemma-4 토크나이저는 `<|turn>user` / `<|turn>model` 형식을 사용하므로  
`train_on_responses_only`가 올바르게 동작하려면 반드시 `apply_chat_template`으로 재포맷해야 합니다.

> ⚠️ **주의**: 섹션 2의 두 번째 셀(`formatting_prompts_func`)은 **섹션 3 실행 후** 실행하세요.  
> `tokenizer` 변수가 없으면 NameError가 발생합니다.


In [2]:
import json, re
from datasets import Dataset

# ── JSONL 로드 ────────────────────────────────────────────────────
records = []
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"총 샘플 수: {len(records)}")
print()
print("=== 원본 text 필드 앞 300자 (ChatML 포맷) ===")
print(records[0]["text"][:300])

# ── <|im_start|> → conversations 구조 파싱 ─────────────────────────
ROLE_MAP = {"system": "system", "user": "user", "assistant": "assistant"}

def parse_im_start(text):
    """<|im_start|>role\ncontent<|im_end|> → [{role, content}, ...]"""
    pattern = re.compile(r"<\|im_start\|>(\w+)\n(.*?)<\|im_end\|>", re.DOTALL)
    return [
        {"role": ROLE_MAP.get(m.group(1), m.group(1)),
         "content": m.group(2).strip()}
        for m in pattern.finditer(text)
    ]

conversations = [parse_im_start(r["text"]) for r in records]

print(f"\n=== 파싱 결과 (샘플 0) ===")
for turn in conversations[0]:
    preview = turn["content"][:80].replace("\n", " ")
    print(f"[{turn['role']}] {preview}...")


총 샘플 수: 865

=== 원본 text 필드 앞 300자 (ChatML 포맷) ===
<|im_start|>system

당신은 국제 무역 법률 및 관습에 정통한 전문 변호사입니다. 사용자가 제공하는 무역 계약서나 서류를 검토하여 전문 법리 보고서를 작성하십시오.

### 🧠 [분석 가이드라인: <thought> 태그 필수]
모든 답변 전에 반드시 <thought> 태그 내에 다음의 4단계 폴백 추론 과정을 거쳐야 합니다.
1. [1단계: 직접 조문] 해당 상황에 직접 적용되는 Article/조 번호 특정.
2. [2단계: 유사 유추] 직접 조문이 없는 경우 유사 조문 및 법리 유추.
3. [3단계: 상위 원칙] 신

=== 파싱 결과 (샘플 0) ===
[system] 당신은 국제 무역 법률 및 관습에 정통한 전문 변호사입니다. 사용자가 제공하는 무역 계약서나 서류를 검토하여 전문 법리 보고서를 작성하십시오. ...
[user] [지시사항] 아래 계약서 초안의 법적 리스크를 분석하라.  [입력 데이터] 【계약 개요】 - 준거법 후보: English law (SGA 197...
[assistant] <thought> 【계약서 사전 검토 보고서】  [1단계: 직접 조문] 적용 법령 확정 - 관할 판단: 준거법이 English law로 명시되어...


## 섹션 3. 모델 로드 및 LoRA 설정

In [3]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name                 = MODEL_ID,
    dtype                      = None,           # 자동 감지 (bf16 권장)
    max_seq_length             = MAX_SEQ_LENGTH,
    load_in_4bit               = LOAD_IN_4BIT,
    full_finetuning            = False,
    use_gradient_checkpointing = "unsloth",      # VRAM 절약 + 긴 컨텍스트
)
print(f"✅ 모델 로드 완료: {MODEL_ID}")
print(f"   MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH:,}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

✅ 모델 로드 완료: unsloth/gemma-4-E2B-it
   MAX_SEQ_LENGTH = 16,384


In [4]:
# LoRA 어댑터 설정 (텍스트 전용)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # E2B 텍스트 전용 파인튜닝
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = 0,
    bias                       = "none",
    random_state               = SEED,
)
model.print_trainable_parameters()


trainable params: 50,675,712 || all params: 5,173,853,728 || trainable%: 0.9795


In [5]:
# 채팅 템플릿 설정 — 반드시 적용
# gemma-4-thinking: 데이터셋에 <thought> 태그 포함됨
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4-thinking")
print("✅ 채팅 템플릿 설정 완료: gemma-4-thinking")
print()
print("→ 다음 단계: 섹션 2의 두 번째 셀(formatting_prompts_func)로 돌아가서 실행")
print("→ 이후 섹션 4 (학습 실행) 진행")


✅ 채팅 템플릿 설정 완료: gemma-4-thinking

→ 다음 단계: 섹션 2의 두 번째 셀(formatting_prompts_func)로 돌아가서 실행
→ 이후 섹션 4 (학습 실행) 진행


In [6]:
# ── apply_chat_template 적용 → Dataset 생성 ──────────────────────
# ⚠️  이 셀은 섹션 3 (모델 로드 + 채팅 템플릿) 실행 후에 실행하세요.

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize              = False,
            add_generation_prompt = False,
        ).removeprefix("<bos>")
        for convo in examples["conversations"]
    ]
    return {"text": texts}

raw_dataset = Dataset.from_list([{"conversations": c} for c in conversations])
dataset     = raw_dataset.map(formatting_prompts_func, batched=True)

print("=== 재포맷 결과 (샘플 0 앞 400자) ===")
print(dataset[0]["text"][:400])
print()
print("✅ <|turn>user / <|turn>model 형식으로 변환 완료")

# ── Shuffle → Train / Eval 분할 ───────────────────────────────────
dataset_shuffled = dataset.shuffle(seed=SEED)
eval_size        = max(1, int(len(dataset_shuffled) * EVAL_RATIO))
train_size       = len(dataset_shuffled) - eval_size
train_dataset    = dataset_shuffled.select(range(train_size))
eval_dataset     = dataset_shuffled.select(range(train_size, len(dataset_shuffled)))

print(f"\n전체: {len(dataset_shuffled)}  학습: {len(train_dataset)}  검증: {len(eval_dataset)}")

# ── 판정 등급 분포 확인 ────────────────────────────────────────────
def count_verdicts(ds, name):
    counts = {"R": 0, "Y": 0, "G": 0}
    markers = {"R": "🔴", "Y": "🟡", "G": "🟢"}
    for r in ds:
        for key, emoji in markers.items():
            if emoji in r["text"]:
                counts[key] += 1
                break
    total = len(ds)
    r_pct = counts["R"]/total*100
    y_pct = counts["Y"]/total*100
    g_pct = counts["G"]/total*100
    print(f"{name}: 🔴 {counts['R']}({r_pct:.0f}%) 🟡 {counts['Y']}({y_pct:.0f}%) 🟢 {counts['G']}({g_pct:.0f}%)")

count_verdicts(train_dataset, "Train")
count_verdicts(eval_dataset,  "Eval ")


Map:   0%|          | 0/865 [00:00<?, ? examples/s]

=== 재포맷 결과 (샘플 0 앞 400자) ===
<|turn>system
당신은 국제 무역 법률 및 관습에 정통한 전문 변호사입니다. 사용자가 제공하는 무역 계약서나 서류를 검토하여 전문 법리 보고서를 작성하십시오.

### 🧠 [분석 가이드라인: <thought> 태그 필수]
모든 답변 전에 반드시 <thought> 태그 내에 다음의 4단계 폴백 추론 과정을 거쳐야 합니다.
1. [1단계: 직접 조문] 해당 상황에 직접 적용되는 Article/조 번호 특정.
2. [2단계: 유사 유추] 직접 조문이 없는 경우 유사 조문 및 법리 유추.
3. [3단계: 상위 원칙] 신의성실, 계약준수, 공서양속 등 법의 일반 원칙 적용.
4. [4단계: 실무 관행] Incoterms, ISBP 등 국제 표준 무역 관행 기반 판단.

[법령 위계 및 관계 설정]
- 위계:

✅ <|turn>user / <|turn>model 형식으로 변환 완료

전체: 865  학습: 822  검증: 43
Train: 🔴 822(100%) 🟡 0(0%) 🟢 0(0%)
Eval : 🔴 43(100%) 🟡 0(0%) 🟢 0(0%)


In [7]:
# ── apply_chat_template 적용 → Dataset 생성 ─────────────────
# tokenizer가 로드된 이후에 이 셀을 실행하세요 (섹션 3 실행 후)
# 재포맷 후 text 필드는 <|turn>user / <|turn>model 형식이 됩니다

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize              = False,
            add_generation_prompt = False,
        ).removeprefix("<bos>")
        for convo in examples["conversations"]
    ]
    return {"text": texts}

raw_dataset = Dataset.from_list([{"conversations": c} for c in conversations])
dataset     = raw_dataset.map(formatting_prompts_func, batched=True)

print("=== 재포맷 결과 (샘플 0 앞 300자) ===")
print(dataset[0]["text"][:300])
print()
print("✅ <|turn>user / <|turn>model 형식으로 변환 완료")
print("   train_on_responses_only가 <|turn>model 마커를 정확히 찾을 수 있습니다.")
print()

# ── Shuffle → Train / Eval 분할 ────────────────────────────────
dataset_shuffled = dataset.shuffle(seed=SEED)

eval_size  = max(1, int(len(dataset_shuffled) * EVAL_RATIO))
train_size = len(dataset_shuffled) - eval_size

train_dataset = dataset_shuffled.select(range(train_size))
eval_dataset  = dataset_shuffled.select(range(train_size, len(dataset_shuffled)))

print(f"전체 샘플  : {len(dataset_shuffled)}개")
print(f"학습 데이터: {len(train_dataset)}개")
print(f"검증 데이터: {len(eval_dataset)}개")
print()

# ── 판정 등급 분포 확인 ─────────────────────────────────────────
def count_verdicts(ds, name):
    counts = {'🔴': 0, '🟡': 0, '🟢': 0}
    for r in ds:
        for k in counts:
            if k in r['text']:
                counts[k] += 1
                break
    total = len(ds)
    print(f"{name}: 🔴 {counts['🔴']}({counts['🔴']/total*100:.0f}%) "
          f"🟡 {counts['🟡']}({counts['🟡']/total*100:.0f}%) "
          f"🟢 {counts['🟢']}({counts['🟢']/total*100:.0f}%)")

count_verdicts(train_dataset, 'Train')
count_verdicts(eval_dataset,  'Eval ')


Map:   0%|          | 0/865 [00:00<?, ? examples/s]

=== 재포맷 결과 (샘플 0 앞 300자) ===
<|turn>system
당신은 국제 무역 법률 및 관습에 정통한 전문 변호사입니다. 사용자가 제공하는 무역 계약서나 서류를 검토하여 전문 법리 보고서를 작성하십시오.

### 🧠 [분석 가이드라인: <thought> 태그 필수]
모든 답변 전에 반드시 <thought> 태그 내에 다음의 4단계 폴백 추론 과정을 거쳐야 합니다.
1. [1단계: 직접 조문] 해당 상황에 직접 적용되는 Article/조 번호 특정.
2. [2단계: 유사 유추] 직접 조문이 없는 경우 유사 조문 및 법리 유추.
3. [3단계: 상위 원칙] 신의성실, 계

✅ <|turn>user / <|turn>model 형식으로 변환 완료
   train_on_responses_only가 <|turn>model 마커를 정확히 찾을 수 있습니다.

전체 샘플  : 865개
학습 데이터: 822개
검증 데이터: 43개

Train: 🔴 822(100%) 🟡 0(0%) 🟢 0(0%)
Eval : 🔴 43(100%) 🟡 0(0%) 🟢 0(0%)


## 섹션 4. 학습 실행

In [8]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        num_train_epochs            = NUM_EPOCHS if MAX_STEPS is None else 1,
        max_steps                   = MAX_STEPS if MAX_STEPS is not None else -1,
        learning_rate               = LEARNING_RATE,
        logging_steps               = 10,
        # ── 검증 설정 ────────────────────────────────────────────
        eval_strategy               = "steps",
        eval_steps                  = EVAL_STEPS,
        per_device_eval_batch_size  = 1,
        load_best_model_at_end      = True,    # eval_loss 최저 시점 자동 선택
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        # ── 체크포인트 — 로컬만 저장 (드라이브 미저장) ──────────
        save_strategy               = "steps",
        save_steps                  = EVAL_STEPS,
        save_total_limit            = 2,       # 최신 2개만 유지 (로컬 용량 절약)
        output_dir                  = str(CHECKPOINT_DIR),   # 로컬 경로
        # ── 기타 ─────────────────────────────────────────────────
        packing                     = True,
        optim                       = "adamw_8bit",
        weight_decay                = 0.05,
        lr_scheduler_type           = "cosine",
        seed                        = SEED,
        report_to                   = "none",
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        dataloader_num_workers      = 0,       # Colab 환경 안정성
    ),
)


print("✅ Trainer 설정 완료")
print(f"   학습: {len(train_dataset)}개 / 검증: {len(eval_dataset)}개")
print(f"   검증 주기: {EVAL_STEPS} 스텝마다")
print(f"   체크포인트: {CHECKPOINT_DIR} (로컬 임시)")


Unsloth: Sample packing skipped (processor-based model detected).


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/822 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/43 [00:00<?, ? examples/s]

✅ Trainer 설정 완료
   학습: 822개 / 검증: 43개
   검증 주기: 30 스텝마다
   체크포인트: /content/work/checkpoints_0514_2352 (로컬 임시)


In [9]:
# ── 패킹 무결성 검증 (보완 버전) ──────────────────────────────
# 1. 단일 샘플이 아닌 실제 DataLoader가 뱉는 '패킹된 배치'를 확인해야 함
batch = next(iter(trainer.get_train_dataloader()))
sample_input_ids = batch["input_ids"][0]
sample_labels = batch["labels"][0]

# 2. 토큰별 상세 매핑 확인 (처음 100개 토큰 대상)
print(f"{'Token Index':<12} | {'Input Token':<20} | {'Label ID':<10}")
print("-" * 50)
# 시퀀스 길이가 100 미만일 수 있으므로 min() 사용
for i in range(min(100, len(sample_input_ids))):
    token = tokenizer.decode([sample_input_ids[i].item()])
    label = sample_labels[i].item()
    print(f"{i:<12} | {token:<20} | {label:<10}")

# 3. 샘플 경계(EOS) 확인
# 패킹 시 EOS 이후 다음 샘플의 시작이 마스킹(-100)되는지 체크
for i, t_id in enumerate(sample_input_ids):
    if t_id.item() == tokenizer.eos_token_id:
        # EOS 토큰이 마지막 인덱스인 경우 IndexError 방지
        if i + 1 < len(sample_labels):
            print(f"\n[경계 발견] Index {i}: EOS 토큰 뒤의 라벨 -> {sample_labels[i+1].item()}")
        else:
            print(f"\n[경계 발견] Index {i}: EOS 토큰이 시퀀스의 맨 마지막에 있습니다.")
        break

Token Index  | Input Token          | Label ID  
--------------------------------------------------
0            | <|turn>              | 105       
1            | system               | 9731      
2            | 
                    | 107       
3            | 당                    | 238749    
4            | 신                    | 238144    
5            | 은                    | 237456    
6            |  국제                  | 150343    
7            |  무                   | 21512     
8            | 역                    | 238726    
9            |  법                   | 64303     
10           | 률                    | 241295    
11           |  및                   | 24566     
12           |  관                   | 18965     
13           | 습                    | 237979    
14           | 에                    | 237223    
15           |  정                   | 12048     
16           | 통                    | 238661    
17           | 한                    | 237384    
18           |  전문

In [10]:
# ── 학습 실행 ────────────────────────────────────────────────────
# E2B loss는 13~15가 정상입니다 (multimodal 모델 특성 — Unsloth 공식 확인)
print("학습 시작...")
print(f"MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH:,}  |  EPOCHS = {NUM_EPOCHS}  |  LR = {LEARNING_RATE}")
trainer_stats = trainer.train()

print(f"\n✅ 학습 완료")
print(f"   학습 시간: {trainer_stats.metrics['train_runtime']:.0f}초")
print(f"   최종 loss: {trainer_stats.metrics['train_loss']:.4f}")
print("   ※ E2B loss 13~15는 정상입니다 (Unsloth 공식 문서)")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


학습 시작...
MAX_SEQ_LENGTH = 16,384  |  EPOCHS = 5  |  LR = 0.0001


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 822 | Num Epochs = 5 | Total steps = 1,030
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 50,675,712 of 5,173,853,728 (0.98% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
30,1.045986,2.491720
60,0.543959,2.131569
90,0.315033,2.069746
120,0.243986,2.029840
150,0.235469,2.046717
180,0.211927,2.038325
210,0.231975,2.058701
240,0.188643,2.077442
270,0.195376,2.049160
300,0.189298,2.036355


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/work/checkpoints_0514_2352/checkpoint-30/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/work/checkpoints_0514_2352/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/work/checkpoints_0514_2352/checkpoint-90/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/work/checkpoints_0514_2352/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/work/checkpoints_0514_2352/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/work/checkpoints_0514_2352/checkpoint-180/tokenizer_config.json.
Unsloth: Re


✅ 학습 완료
   학습 시간: 5037초
   최종 loss: 0.2384
   ※ E2B loss 13~15는 정상입니다 (Unsloth 공식 문서)


## 섹션 5. 저장

### 흐름
```
[학습 완료 모델]
      │
      ├─── 5A: 어댑터 저장 → drive/adapter_{RUN_ID}/   ← ✅ 드라이브 (~100MB)
      │
      └─── 5B: 병합 → 로컬 merged_16bit/ (임시)
                  └─── 5C: GGUF 4bit 변환 → drive/gguf_{RUN_ID}/  ← ✅ 드라이브
                              └─── 5D: merged_16bit 삭제 (용량 확보)
```


In [11]:
# ── 5A: 어댑터 저장 → 드라이브 ✅ ──────────────────────────────
# load_best_model_at_end=True이므로 현재 model은 이미 최적 체크포인트 상태

print(f"어댑터 저장 중: {ADAPTER_DIR}")
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

adapter_size = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file())
print(f"✅ 어댑터 저장 완료 — {adapter_size / 1e6:.1f} MB")
print(f"   경로: {ADAPTER_DIR}")


어댑터 저장 중: /content/drive/MyDrive/finetune/output/adapter_0514_2352


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/finetune/output/adapter_0514_2352/tokenizer_config.json.


✅ 어댑터 저장 완료 — 235.0 MB
   경로: /content/drive/MyDrive/finetune/output/adapter_0514_2352


In [12]:
# ── 5B: 모델 병합 → 로컬 임시 (드라이브 미저장) ────────────────
# 병합된 16bit 풀웨이트는 GGUF 변환용 임시 파일
# 드라이브에 저장하면 수십 GB 소모 → 로컬에만 저장

print(f"모델 병합 중 (로컬 임시): {MERGED_16BIT_DIR}")
print("  ※ 이 파일은 드라이브에 저장되지 않습니다 (런타임 종료 시 삭제)")

model.save_pretrained_merged(
    str(MERGED_16BIT_DIR),
    tokenizer,
    save_method = "merged_16bit",
)

merged_size = sum(f.stat().st_size for f in MERGED_16BIT_DIR.rglob("*") if f.is_file())
print(f"✅ 병합 완료 — {merged_size / 1e9:.2f} GB (로컬 임시)")


모델 병합 중 (로컬 임시): /content/work/merged_16bit
  ※ 이 파일은 드라이브에 저장되지 않습니다 (런타임 종료 시 삭제)


config.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/work/merged_16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:27<00:00, 27.32s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:06<00:00, 66.50s/it]


Unsloth: Merge process complete. Saved to `/content/work/merged_16bit`
✅ 병합 완료 — 10.28 GB (로컬 임시)


In [19]:
# ── 5C: 4bit GGUF 변환 → 로컬 임시 저장 ──────────────────────
# q4_k_m: 품질-용량 균형이 가장 좋은 표준 4bit 양자화 방식
# 변환 소스: 로컬 merged_16bit (임시)

print(f"GGUF 변환 중 (q4_k_m) (로컬 임시): {LOCAL_GGUF_DIR}")

model.save_pretrained_gguf(
    str(LOCAL_GGUF_DIR), # Save to local temporary directory
    tokenizer,
    quantization_method = "q4_k_m",
)

gguf_files = list(LOCAL_GGUF_DIR.glob("*.gguf")) # List from local temp dir
for f in gguf_files:
    print(f"  {f.name}: {f.stat().st_size / 1e9:.2f} GB")

print(f"\n✅ GGUF 변환 완료 (로컬 임시)")
print(f"   저장 위치: {LOCAL_GGUF_DIR}")

GGUF 변환 중 (q4_k_m) (로컬 임시): /content/work/gguf_tmp
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /content/work/gguf_tmp/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:26<00:00, 26.86s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:09<00:00, 69.23s/it]


Unsloth: Merge process complete. Saved to `/content/work/gguf_tmp`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/work/gguf_tmp_gguf/gemma-4-e2b-it.BF16.gguf', '/content/work/gguf_tmp_gguf/gemma-4-e2b-it.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/work/gguf_tmp_g

In [20]:
# ── 5D: 로컬 임시 파일 정리 및 드라이브로 GGUF 이동 ─────────────────
import shutil
import glob

# 병합 16bit 임시 파일 삭제
shutil.rmtree(str(MERGED_16BIT_DIR), ignore_errors=True)
print(f"🗑️  병합 16bit 임시 파일 삭제 완료: {MERGED_16BIT_DIR}")

# 로컬 임시 GGUF 파일을 드라이브로 이동
print(f"GGUF 파일을 드라이브로 이동 중: {LOCAL_GGUF_DIR} -> {GGUF_DIR}")
local_gguf_files = glob.glob(str(LOCAL_GGUF_DIR / "*.gguf"))
if local_gguf_files:
    for local_path in local_gguf_files:
        filename = os.path.basename(local_path)
        drive_path = GGUF_DIR / filename
        shutil.move(local_path, drive_path)
        print(f"  ✅ {filename} 이동 완료")
    shutil.rmtree(str(LOCAL_GGUF_DIR))
    print(f"🗑️  GGUF 임시 폴더 삭제 완료: {LOCAL_GGUF_DIR}")
else:
    print("⚠️  이동할 GGUF 파일이 없습니다.")

# 체크포인트 정리 (선택 — 주석 해제 시 삭제)
# shutil.rmtree(str(CHECKPOINT_DIR), ignore_errors=True)
# print(f"🗑️  체크포인트 삭제 완료")

print()
print("=== 드라이브 저장 결과 요약 ===")
for d, label in [(ADAPTER_DIR, "어댑터(safetensor)"), (GGUF_DIR, "GGUF 4bit (q4_k_m)")]:
    try:
        size = sum(f.stat().st_size for f in d.rglob("*") if f.is_file())
        print(f"  {label}: {d.name}  ({size / 1e9:.2f} GB)")
    except Exception as e:
        print(f"   {label}: 경로 확인 필요 ({e})")

🗑️  병합 16bit 임시 파일 삭제 완료: /content/work/merged_16bit
GGUF 파일을 드라이브로 이동 중: /content/work/gguf_tmp -> /content/drive/MyDrive/finetune/output/gguf_0515_0151
⚠️  이동할 GGUF 파일이 없습니다.

=== 드라이브 저장 결과 요약 ===
  어댑터(safetensor): adapter_0515_0151  (0.00 GB)
  GGUF 4bit (q4_k_m): gguf_0515_0151  (0.00 GB)


In [21]:
# ── 최종 결과 요약 ────────────────────────────────────────────────
print("=" * 55)
print("           학습 및 저장 완료 요약")
print("=" * 55)
print(f" Run ID         : {RUN_ID}")
print(f" MAX_SEQ_LENGTH : {MAX_SEQ_LENGTH:,} 토큰")
print(f" LORA RANK      : {LORA_RANK}")
print()
print(" [드라이브 저장 파일]")
for d, label in [(ADAPTER_DIR, "어댑터(safetensor)"), (GGUF_DIR, "GGUF 4bit (q4_k_m)")]:
    try:
        size = sum(f.stat().st_size for f in d.rglob("*") if f.is_file())
        print(f"   {label}")
        print(f"   경로: {d}")
        print(f"   크기: {size / 1e9:.2f} GB")
        print()
    except Exception as e:
        print(f"   {label}: 경로 확인 필요 ({e})")
print(" [드라이브 미저장 (로컬 삭제됨)]")
print(f"   체크포인트 : {CHECKPOINT_DIR}")
print(f"   병합 16bit : {MERGED_16BIT_DIR}")
print("=" * 55)


           학습 및 저장 완료 요약
 Run ID         : 0515_0151
 MAX_SEQ_LENGTH : 16,384 토큰
 LORA RANK      : 32

 [드라이브 저장 파일]
   어댑터(safetensor)
   경로: /content/drive/MyDrive/finetune/output/adapter_0515_0151
   크기: 0.00 GB

   GGUF 4bit (q4_k_m)
   경로: /content/drive/MyDrive/finetune/output/gguf_0515_0151
   크기: 0.00 GB

 [드라이브 미저장 (로컬 삭제됨)]
   체크포인트 : /content/work/checkpoints_0515_0151
   병합 16bit : /content/work/merged_16bit
